# Train and profile the GRPO pipeline

Start with the local fixture. This notebook orchestrates functions from Python modules.
Use the Python 3.12 project kernel. Hosted installation and durable-save instructions are
in `docs/NOTEBOOK_RUNTIME_AND_RESUME.md`. Avoid initializing CUDA in this notebook before
a multi-GPU launch. The default run performs real GRPO updates with artificial rewards.

In [ ]:
import os
from dataclasses import replace
from pathlib import Path

from rl_pipeline.config import RunConfig
from rl_pipeline.train import default_trainer_options, train_worker

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "pyproject.toml").exists():
    PROJECT_DIR = PROJECT_DIR.parent
assert (PROJECT_DIR / "pyproject.toml").exists(), "Run from the project or notebooks directory."
os.chdir(PROJECT_DIR)

# Run identity and compute. Use one output folder per independent seed.
RUN_NAME = "smoke"
SEEDS = [42]
GAME_SEED = 123
USE_CPU = True
NUM_PROCESSES = 1
PRECISION = "no"  # "fp16" for T4 GPUs; "bf16" only on supported GPUs.
RESUME = False

# Model and task. None builds a tiny local Transformer without downloads.
MODEL_NAME = None
MODEL_REVISION = "main"  # Pin a commit when using a downloaded policy.
LORA_RANK = 4
LORA_ALPHA = 8
LORA_TARGETS = ()  # Tiny GPT2: c_attn. Qwen default: q_proj and v_proj.
DATASET_PATH = str(PROJECT_DIR / "tests" / "fixtures" / "animals.json")
GAME_COUNT = 8
MAX_MOVES = 6
MAX_ACTION_TOKENS = 1  # Use a representative value, such as 64, for a real policy.
MAX_CONTEXT_TOKENS = 2048
MAX_COMPLETION_TOKENS = 128  # Entire continuation, including supplied feedback.

# Parsing and scoring. Live credentials belong in notebook secrets/environment.
PARSER_MODEL = None  # An exact Anthropic model ID enables the API adapter.
FREE_FORM_QUESTIONS = False
PARSER_CONCURRENCY = 2  # Shared across the workers of this run.
PARSER_REQUESTS_PER_MINUTE = 30
PARSER_MAX_ATTEMPTS = 2
REWARD = "fixture"  # Switch to "task" for success/move cost; information remains zero.
MOVE_COST = 0.02

# Optimization and GRPO. All other native options remain visible in train.py.
GROUP_SIZE = 4
GROUPS_PER_MICROBATCH = 1
GRADIENT_ACCUMULATION_STEPS = 1
MAX_STEPS = 2
LEARNING_RATE = 1e-3
LR_SCHEDULER = "constant"
WARMUP_STEPS = 0
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
TEMPERATURE = 1.0
KL_COEFFICIENT = 0.0
CLIP_EPSILON = 0.2
LOSS_TYPE = "grpo"
SCALE_REWARDS = "group"
GRADIENT_CHECKPOINTING = False

# Recovery and logs. Configure a real persistent destination before costly runs.
BACKUP_DIR = None  # Example: a dedicated mounted Drive folder.
HUB_REPO = None  # Existing private owner/model-repo; HF_TOKEN comes from secrets.
BACKUP_ROLLOUTS = True
SAVE_STEPS = 1
KEEP_CHECKPOINTS = 2
LOGGING_STEPS = 1
TENSORBOARD_FLUSH_SECONDS = 10
PROFILE_WARMUP_UPDATES = 0
PLANNED_FULL_ACTION_TOKENS = None

## Assemble the settings

Each local microbatch contains whole groups. A fresh generation batch supplies one optimizer
update, including every accumulation microbatch. The starter deliberately uses one update
pass per fresh batch and untruncated temperature sampling. For full-load profiling, start
with several warm-up updates and a longer measured window; the fixture timings are not
estimates for Qwen or the live parser.

In [ ]:
run = RunConfig(
    dataset_path=DATASET_PATH,
    model_name=MODEL_NAME,
    model_revision=MODEL_REVISION,
    game_count=GAME_COUNT,
    game_seed=GAME_SEED,
    max_moves=MAX_MOVES,
    max_action_tokens=MAX_ACTION_TOKENS,
    max_context_tokens=MAX_CONTEXT_TOKENS,
    reward=REWARD,
    move_cost=MOVE_COST,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_targets=LORA_TARGETS,
    parser_model=PARSER_MODEL,
    parser_concurrency=PARSER_CONCURRENCY,
    parser_requests_per_minute=PARSER_REQUESTS_PER_MINUTE,
    parser_max_attempts=PARSER_MAX_ATTEMPTS,
    free_form_questions=FREE_FORM_QUESTIONS,
    backup_dir=BACKUP_DIR,
    hub_repo=HUB_REPO,
    backup_rollouts=BACKUP_ROLLOUTS,
    tensorboard_flush_seconds=TENSORBOARD_FLUSH_SECONDS,
)

trainer_options = default_trainer_options("runs/placeholder")
trainer_options.update(
    max_steps=MAX_STEPS,
    per_device_train_batch_size=GROUP_SIZE * GROUPS_PER_MICROBATCH,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=GROUP_SIZE,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    temperature=TEMPERATURE,
    beta=KL_COEFFICIENT,
    epsilon=CLIP_EPSILON,
    loss_type=LOSS_TYPE,
    scale_rewards=SCALE_REWARDS,
    max_completion_length=MAX_COMPLETION_TOKENS,
    use_cpu=USE_CPU,
    fp16=PRECISION == "fp16",
    bf16=PRECISION == "bf16",
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    save_steps=SAVE_STEPS,
    save_total_limit=KEEP_CHECKPOINTS,
    logging_steps=LOGGING_STEPS,
)
assert not (USE_CPU and PRECISION != "no"), "Use full precision for the CPU fixture."

## Launch

For a DDP GPU run, all workers train one policy. SEEDS launches independent policies.
Persistent folders must also be distinct per seed. Resume restores the same planned run;
keep its settings unchanged and set RESUME=True after restoring its output directory.

In [ ]:
from accelerate import notebook_launcher

from integration.artifacts import write_json

completed_runs = []
for seed in SEEDS:
    output = PROJECT_DIR / "runs" / f"{RUN_NAME}-seed-{seed}"
    options = dict(trainer_options)
    options.update(
        output_dir=str(output),
        logging_dir=str(output / "tensorboard"),
        seed=seed,
        data_seed=seed,
    )
    seed_run = replace(
        run,
        backup_dir=str(Path(BACKUP_DIR) / output.name) if BACKUP_DIR else None,
    )
    write_json(output / "notebook_settings.json", {"run": seed_run, "trainer_options": options})
    notebook_launcher(
        train_worker,
        args=(seed_run, options, "latest" if RESUME else None),
        num_processes=NUM_PROCESSES,
        mixed_precision=PRECISION,
    )
    completed_runs.append(output)

## Read saved training diagnostics

This is compute profiling, not scientific evaluation. Update time includes rollout waiting;
checkpoint publishing is added separately. Startup and final export are separate costs.
Inspect per-wave timing in the saved turn JSONL to identify API bottlenecks.

In [ ]:
from rl_pipeline.profiling import summarize_run

for output in completed_runs:
    profile = summarize_run(output, warmup_updates=PROFILE_WARMUP_UPDATES)
    print(output.name, profile)
    if PLANNED_FULL_ACTION_TOKENS is not None:
        estimated_hours = PLANNED_FULL_ACTION_TOKENS / profile["action_tokens_per_second"] / 3600
        print("Estimated training hours:", estimated_hours)
        print("Estimated allocated GPU-hours:", estimated_hours * NUM_PROCESSES)

## TensorBoard

From a terminal: `uv run --locked --extra cpu tensorboard --logdir runs`.
In Colab, use the platform's TensorBoard extension after launching training.
Scalar events are backed by structured metric files. Checkpoints, not TensorBoard,
are the source of training recovery. Exact evaluation metrics belong in the next notebook.